# HPO LunarLander

Optuna hyperparameter optimization for `VectorTrainer` on LunarLander.

The happy path on Colab is labeled HP1 to HP4.

Hardware: L4 GPU is the intended runtime.

## Setup

### Runtime setup

Set up the runtime by running exactly one code cell in this section.

#### Colab

In [ ]:
# HP1
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys

from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.lunar_lander.logging import configure_file_logging

study_dir = Path("/content/drive/MyDrive/rl_lab/hpo")
drive.mount("/content/drive")
study_dir.mkdir(parents=True, exist_ok=True)
configure_file_logging(study_dir)

HPO_STUDY_DIR = study_dir

#### Local

In [ ]:
# Local setup
from pathlib import Path
import sys
sys.path.insert(0, "../../dqn/src")
sys.path.insert(0, "../src")
HPO_STUDY_DIR = Path("../runs")
HPO_STUDY_DIR.mkdir(parents=True, exist_ok=True)


## Imports

In [ ]:
# HP2
from dataclasses import replace

import torch

from dqn.vector_training import VectorTrainingConfig
from hpo.evaluation.scoring import ScoringConfig
from hpo.lunar_lander.environment import LunarLanderEnvironmentFactory
from hpo.objective import TrialConfig
from hpo.study import StudyRunner, neighbors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRIAL_CFG = TrialConfig(device=device)
ENVIRONMENT_FACTORY = LunarLanderEnvironmentFactory()
SCORING_CFG = ScoringConfig()
device

## Optimization

### Define SearchSpaces

In [ ]:
# HP3
BATCH_SIZES = [256, 512, 1024]
LEARNING_STARTS = [2_500, 5_000, 10_000]
OPTIMIZE_EVERY = [2, 4, 8]
REPLAY_CAPACITIES = [50_000, 100_000, 200_000]


BASELINE_PARAMS = {
    "learning_rate": 3e-4,
    "batch_size": 512,
    "eps_end": 0.05,
    "eps_decay": 25_000,
    "gamma": 0.99,
    "tau": 0.005,
    "learning_starts": 5_000,
    "optimize_every": 4,
    "replay_memory_capacity": 100_000,
}


def vector_config(
    *,
    learning_rate,
    batch_size,
    eps_end,
    eps_decay,
    learning_starts,
    optimize_every,
):
    return VectorTrainingConfig(
        num_episodes=600,
        batch_size=batch_size,
        gamma=0.99,
        eps_start=1.0,
        eps_end=eps_end,
        eps_decay=eps_decay,
        tau=0.005,
        learning_rate=learning_rate,
        learning_starts=learning_starts,
        optimize_every=optimize_every,
    )


class SearchSpace0:
    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

    def training_config(self, trial, params):
        return vector_config(
            learning_rate=params["learning_rate"],
            batch_size=params["batch_size"],
            eps_end=params["eps_end"],
            eps_decay=params["eps_decay"],
            learning_starts=params["learning_starts"],
            optimize_every=params["optimize_every"],
        )


class SearchSpace1:
    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

    def training_config(self, trial, params):
        return vector_config(
            learning_rate=trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True),
            batch_size=trial.suggest_categorical("batch_size", BATCH_SIZES),
            eps_end=params["eps_end"],
            eps_decay=params["eps_decay"],
            learning_starts=trial.suggest_categorical("learning_starts", LEARNING_STARTS),
            optimize_every=trial.suggest_categorical("optimize_every", OPTIMIZE_EVERY),
        )


class SearchSpace2:
    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

    def training_config(self, trial, params):
        return vector_config(
            learning_rate=params["learning_rate"],
            batch_size=params["batch_size"],
            eps_end=trial.suggest_float("eps_end", 0.01, 0.10),
            eps_decay=trial.suggest_int("eps_decay", 10_000, 60_000, log=True),
            learning_starts=params["learning_starts"],
            optimize_every=params["optimize_every"],
        )


class SearchSpace3:
    def replay_memory_capacity(self, trial, params):
        return trial.suggest_categorical("replay_memory_capacity", REPLAY_CAPACITIES)

    def training_config(self, trial, params):
        return vector_config(
            learning_rate=params["learning_rate"],
            batch_size=params["batch_size"],
            eps_end=params["eps_end"],
            eps_decay=params["eps_decay"],
            learning_starts=params["learning_starts"],
            optimize_every=params["optimize_every"],
        )


class SearchSpace4:
    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

    def training_config(self, trial, params):
        eps_end = params["eps_end"]
        eps_decay = params["eps_decay"]
        return vector_config(
            learning_rate=trial.suggest_float(
                "learning_rate",
                params["learning_rate"] / 2,
                params["learning_rate"] * 2,
                log=True,
            ),
            batch_size=trial.suggest_categorical(
                "batch_size",
                neighbors(params["batch_size"], BATCH_SIZES),
            ),
            eps_end=trial.suggest_float(
                "eps_end",
                max(0.01, eps_end - 0.02),
                min(0.10, eps_end + 0.02),
            ),
            eps_decay=trial.suggest_int(
                "eps_decay",
                max(1, eps_decay // 2),
                eps_decay * 2,
                log=True,
            ),
            learning_starts=trial.suggest_categorical(
                "learning_starts",
                neighbors(params["learning_starts"], LEARNING_STARTS),
            ),
            optimize_every=trial.suggest_categorical(
                "optimize_every",
                neighbors(params["optimize_every"], OPTIMIZE_EVERY),
            ),
        )


### Optimize

In [ ]:
# HP4
runner = StudyRunner(
    storage_path=lambda name: HPO_STUDY_DIR / f"{name}.db",
    environment_factory=ENVIRONMENT_FACTORY,
    trial_cfg=TRIAL_CFG,
    incumbent_params=BASELINE_PARAMS,
)

runner.run(
    "s0_qe_baseline", SearchSpace0(), 3, SCORING_CFG, robust=False,
)
study0 = runner.studies[-1]
study0.set_user_attr("baseline_params", BASELINE_PARAMS)
scoring_cfg = replace(
    SCORING_CFG,
    baseline_env_steps=study0.user_attrs["baseline_env_steps"],
    baseline_processed_samples=study0.user_attrs["baseline_processed_samples"],
)

runner.run("s1_qe_update_economy", SearchSpace1(), 40, scoring_cfg)
runner.run("s2_qe_exploration", SearchSpace2(), 40, scoring_cfg)
runner.run("s3_qe_replay_capacity", SearchSpace3(), 10, scoring_cfg)
runner.run("s4_qe_joint_finetune", SearchSpace4(), 30, scoring_cfg)


## Analysis

### Load studies (only after runtime reset)

In [ ]:
# Run only after the runtime environment has been reset.
!pip install -q optuna plotly nbformat

from pathlib import Path
from google.colab import drive
import optuna

drive.mount("/content/drive")
HPO_STUDY_DIR = Path("/content/drive/MyDrive/rl_lab/hpo")

def load_study(name):
    return optuna.load_study(
        study_name=name,
        storage=f"sqlite:///{HPO_STUDY_DIR / f'{name}.db'}",
    )

study0 = load_study("s0_qe_baseline")
study1 = load_study("s1_qe_update_economy")
study2 = load_study("s2_qe_exploration")
study3 = load_study("s3_qe_replay_capacity")
study4 = load_study("s4_qe_joint_finetune")


### Best hyperparameters

In [ ]:
import pandas as pd

def best_params(study):
    return study.user_attrs.get("robust_best_params") or study.best_trial.params

baseline = study0.user_attrs["baseline_params"]
s1 = baseline | best_params(study1)
s2 = s1 | best_params(study2)
s3 = s2 | best_params(study3)
s4 = s3 | best_params(study4)
optimized = [set(best_params(study)) for study in [study1, study2, study3, study4]]

columns = ["S0 Baseline", "S1 Update economy", "S2 Exploration", "S3 Replay capacity", "S4 Joint fine-tune"]
table = pd.DataFrame(dict(zip(columns, [baseline, s1, s2, s3, s4])))
def study_score(study):
    return study.user_attrs.get("robust_best_objective_score", study.best_value)

table.loc["score"] = [study_score(study) for study in [study0, study1, study2, study3, study4]]
optimized_by_study = [set(), *optimized]
fallback_scores = {
    column for column, study in zip(columns, [study0, study1, study2, study3, study4])
    if "robust_best_objective_score" not in study.user_attrs
}

def highlight_optimized(_):
    styles = pd.DataFrame("", index=table.index, columns=table.columns)
    for column, parameters in zip(columns, optimized_by_study):
        styles.loc[list(parameters), column] = "background-color: #fff2cc"
    styles.loc["score", :] = "border-top: 2px solid #888; font-weight: bold"
    styles.loc["score", list(fallback_scores)] += "; color: #777; font-style: italic"
    return styles

integer_rows = ["batch_size", "eps_decay", "learning_starts", "optimize_every", "replay_memory_capacity"]
styled_table = table.style.format("{:.3f}")
styled_table = styled_table.format("{:.6f}", subset=pd.IndexSlice[["learning_rate"], :])
styled_table = styled_table.format("{:.0f}", subset=pd.IndexSlice[integer_rows, :])
styled_table = styled_table.format("{:.3f}", subset=pd.IndexSlice[["score"], :])
styled_table = styled_table.set_table_styles([{
    "selector": "tbody tr:last-child th",
    "props": "border-top: 2px solid #888; font-weight: bold",
}])
display(styled_table.apply(highlight_optimized, axis=None))
print("Gray italic score: best-trial fallback instead of robust score.")


### Plots

Score dependence on the hyperparameters optimized in each study.

In [ ]:
from IPython.display import display
from optuna.visualization import plot_contour, plot_slice

print("S1: Update economy")
display(plot_contour(study1, target_name="Score"))
print("S2: Exploration")
display(plot_contour(study2, target_name="Score"))
print("S3: Replay capacity")
display(plot_slice(study3, target_name="Score"))
print("S4: Joint fine-tune")
display(plot_contour(study4, target_name="Score"))
